# Notebook 04 - Other AI Domains

Goes with **Lesson 04**. Five small, complete examples.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## 1. Recommender system (collaborative filtering)

Ratings from 1 to 5. Empty (NaN) = the user has not watched that movie yet.
**Idea:** find users with similar taste, and recommend what **they** liked.

In [ ]:
ratings = pd.DataFrame({
    "Action Hero":   [5, 4, np.nan, 1, 5],
    "Space Wars":    [4, 5, 1, np.nan, 4],
    "Love Story":    [1, np.nan, 5, 4, 2],
    "Comedy Nights": [np.nan, 2, 4, 5, 1],
    "Robot Uprising":[np.nan, 5, 2, 1, 4],
    "Family Drama":  [2, 1, 4, np.nan, np.nan],
}, index=["Asha", "Ravi", "Meera", "John", "Priya"])
ratings

In [ ]:
def cosine(a, b):
    return a @ b / (np.linalg.norm(a) * np.linalg.norm(b))

filled = ratings.fillna(0).to_numpy()
users = list(ratings.index)
similarity = pd.DataFrame([[cosine(filled[i], filled[j]) for j in range(len(users))] for i in range(len(users))],
                          index=users, columns=users)
print("How similar is each pair of users (1 = same taste):")
similarity.round(2)

In [ ]:
def recommend(user):
    others = similarity[user].drop(user)
    scores = {}
    for movie in ratings.columns[ratings.loc[user].isna()]:      # only movies the user has NOT rated
        rated_by = ratings[movie].dropna().index.drop(user, errors="ignore")
        weights = others[rated_by]
        scores[movie] = (ratings.loc[rated_by, movie] * weights).sum() / weights.sum()
    return pd.Series(scores).sort_values(ascending=False)

print("Predicted ratings for Asha's unwatched movies:")
print(recommend("Asha").round(2))
print()
print("Recommendation for Asha:", recommend("Asha").index[0])

Asha likes action and space movies, and her taste is closest to Priya and Ravi, who loved **Robot Uprising**.

## 2. Reinforcement learning: Q-learning in a grid world

The agent starts at the top-left `S` and must reach the goal `G` at the bottom-right. Each step costs -1, reaching the goal gives +10. Nobody tells it the path. It learns a **Q-table**: how good each action is in each cell.

In [ ]:
size = 5
actions = {0: (-1, 0), 1: (1, 0), 2: (0, -1), 3: (0, 1)}   # up, down, left, right
action_names = {0: "up", 1: "down", 2: "left", 3: "right"}
goal = (4, 4)

Q = np.zeros((size, size, 4))
alpha, gamma, epsilon, episodes = 0.5, 0.9, 0.2, 500
rng = np.random.default_rng(0)

def take_step(state, action):
    dr, dc = actions[action]
    r = min(max(state[0] + dr, 0), size - 1)
    c = min(max(state[1] + dc, 0), size - 1)
    new_state = (r, c)
    reward = 10 if new_state == goal else -1
    return new_state, reward

for episode in range(episodes):
    state = (0, 0)
    for _ in range(100):
        if rng.random() < epsilon:
            action = int(rng.integers(4))                 # explore
        else:
            action = int(np.argmax(Q[state]))             # exploit
        new_state, reward = take_step(state, action)
        # Q-learning update: move the old estimate towards (reward + best future value)
        Q[state][action] += alpha * (reward + gamma * np.max(Q[new_state]) - Q[state][action])
        state = new_state
        if state == goal:
            break

# Follow the learned policy
state, path = (0, 0), [(0, 0)]
while state != goal and len(path) < 30:
    state, _ = take_step(state, int(np.argmax(Q[state])))
    path.append(state)

grid = [["." for _ in range(size)] for _ in range(size)]
for r, c in path:
    grid[r][c] = "*"
grid[0][0], grid[4][4] = "S", "G"
print("Learned path (", len(path) - 1, "steps - the shortest possible is 8 ):")
for row in grid:
    print(" ".join(row))

## 3. Explainable AI: why did the model decide that?

We train a random forest on a medical dataset that comes with scikit-learn (breast cancer tumours: malignant or benign). Then **permutation importance** shuffles one column at a time: if the score drops a lot, that column was important.

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.model_selection import train_test_split

data = load_breast_cancer(as_frame=True)
X_train, X_test, y_train, y_test = train_test_split(data.data, data.target, test_size=0.25, random_state=42)

forest = RandomForestClassifier(n_estimators=200, random_state=42).fit(X_train, y_train)
print(f"Accuracy: {forest.score(X_test, y_test):.3f}")

result = permutation_importance(forest, X_test, y_test, n_repeats=10, random_state=42)
importance = pd.Series(result.importances_mean, index=data.data.columns).sort_values().tail(8)

plt.figure(figsize=(8, 4))
plt.barh(importance.index, importance.values)
plt.title("Top 8 features the model depends on (permutation importance)")
plt.xlabel("Drop in accuracy when the feature is shuffled")
plt.show()

A doctor can now see **what the model looked at**. Tools like **SHAP** and **LIME** go further and explain **one single** prediction.

## 4. Predictive analytics: forecast next months' sales

In [ ]:
from sklearn.linear_model import LinearRegression

months = np.arange(1, 25)                                    # 2 years of monthly data
rng = np.random.default_rng(3)
sales = 200 + 8 * months + 30 * np.sin(2 * np.pi * months / 12) + rng.normal(0, 8, len(months))

# Features: the trend (month number) and the season (position in the year)
def make_features(m):
    return np.column_stack([m, np.sin(2 * np.pi * m / 12), np.cos(2 * np.pi * m / 12)])

model = LinearRegression().fit(make_features(months), sales)
future = np.arange(25, 31)
forecast = model.predict(make_features(future))

plt.figure(figsize=(9, 4))
plt.plot(months, sales, marker="o", label="past sales")
plt.plot(future, forecast, marker="o", linestyle="--", label="forecast")
plt.title("Monthly sales and a 6-month forecast")
plt.xlabel("Month")
plt.ylabel("Sales (units)")
plt.legend()
plt.show()
print("Forecast for the next 6 months:", forecast.round(0))

## 5. AI in cybersecurity: anomaly detection for logins

We have 300 normal logins and a few strange ones. **Isolation Forest** learns what "normal" looks like and flags unusual rows - without any labels.

In [ ]:
from sklearn.ensemble import IsolationForest

rng = np.random.default_rng(7)
normal = pd.DataFrame({
    "login_hour": rng.normal(13, 3, 300).clip(7, 22).round(),
    "failed_attempts": rng.poisson(0.3, 300),
    "data_downloaded_mb": rng.normal(40, 12, 300).clip(1).round(),
})
strange = pd.DataFrame({
    "login_hour": [3, 2, 4],
    "failed_attempts": [9, 0, 7],
    "data_downloaded_mb": [45, 900, 600],
})
logins = pd.concat([normal, strange], ignore_index=True)

detector = IsolationForest(contamination=0.01, random_state=42)
logins["flag"] = detector.fit_predict(logins[["login_hour", "failed_attempts", "data_downloaded_mb"]])

print("Rows flagged as suspicious (-1):")
logins[logins["flag"] == -1]

The detector found the 3 a.m. logins with many failed attempts or huge downloads. It may also flag one normal-looking row - anomaly detection is not perfect, so a security team checks every flagged row.

**Try it yourself:** add a new user to the ratings table and call `recommend()` for them.

Go back to **Lesson 04** and do the **Check** section.